# Lesson 5: Loops & Validation — Retry Until Valid Input

## Objective
Use conditional edges that point **backwards** to create validation loops that retry until input passes all checks — without crashing, without infinite loops.

## Limitation of Lesson 4
- ❌ Invalid input (unknown operation, divide by zero) was silently accepted or returned errors
- ❌ Graph always moved forward — no way to reject and retry
- ❌ No input correction mechanism

## What is NEW in Lesson 5?
- ✅ **Backwards edges** — a conditional edge can point to an EARLIER node (creates a cycle)
- ✅ Validation node that inspects and rejects bad input
- ✅ Fix-input node that corrects problems automatically
- ✅ Attempt counter to prevent infinite loops
- ✅ Understanding cycles and `GraphRecursionError`

## Theory: Cycles in LangGraph

LangGraph is **NOT a DAG** (Directed Acyclic Graph).  
It supports **cycles** — edges that point backward to earlier nodes.

```
START → validate ──→ [valid?]
                        ├── YES → compute → END
                        └── NO  → fix_input → validate  ← CYCLE!
```

### Rules for safe cycles:
1. **Exit condition**: always have a path that reaches `END`
2. **Attempt counter**: limit the number of iterations
3. **`recursion_limit`**: LangGraph's built-in safety net (default: 25 steps)

If none of these exist → `GraphRecursionError` after 25 steps.

### Why cycles matter in production
- **Retry loops**: retry failed API calls
- **Human-in-the-loop**: wait for approval before proceeding (Lesson 18)
- **LLM reflection**: ask the LLM to check its own answer (Lesson 20)
- **Tool use loops**: LLM calls tools until it has enough information (Lesson 8)


In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, Literal, Annotated
import operator
from IPython.display import Image, display

MAX_ATTEMPTS = 3   # Maximum validation retries before giving up
print(f"✓ Imports successful | MAX_ATTEMPTS = {MAX_ATTEMPTS}")


In [ ]:
# ── Cell 2: State with Validation Fields ─────────────────────────────────────
class State(TypedDict):
    a: int
    b: int
    operation: str
    result: Optional[float]
    is_valid: bool           # Did validation pass this round?
    error_msg: str           # What error was found?
    attempts: int            # How many validation attempts so far?
    history: Annotated[list, operator.add]

print("✓ State with validation fields defined")
print()
print("  is_valid  → True/False: did this round of validation pass?")
print("  error_msg → why it failed")
print("  attempts  → prevents infinite loops")


In [ ]:
# ── Cell 3: Validation Node ─────────────────────────────────────────────────
# Runs BEFORE compute
# Checks all rules and sets is_valid + error_msg

def validate_node(state: State) -> dict:
    a, b, op = state["a"], state["b"], state["operation"]
    attempts = state["attempts"] + 1  # Increment counter
    
    print(f"  [validate] Attempt #{attempts}: a={a}, b={b}, op='{op}'")
    
    # Rule 1: Valid operation?
    if op not in ("add", "multiply", "divide"):
        msg = f"Unknown operation '{op}'. Must be: add, multiply, divide"
        print(f"  [validate] ✗ {msg}")
        return {"is_valid": False, "error_msg": msg, "attempts": attempts}
    
    # Rule 2: Divide by zero?
    if op == "divide" and b == 0:
        msg = "Cannot divide by zero. 'b' must be non-zero for divide."
        print(f"  [validate] ✗ {msg}")
        return {"is_valid": False, "error_msg": msg, "attempts": attempts}
    
    # Rule 3: Numbers in safe range?
    if not (-1000 <= a <= 1000 and -1000 <= b <= 1000):
        msg = f"Numbers must be in range [-1000, 1000]. Got a={a}, b={b}"
        print(f"  [validate] ✗ {msg}")
        return {"is_valid": False, "error_msg": msg, "attempts": attempts}
    
    print(f"  [validate] ✓ All checks passed!")
    return {"is_valid": True, "error_msg": "", "attempts": attempts}

print("✓ Validation node defined (3 rules)")


In [ ]:
# ── Cell 4: Fix-Input Node ───────────────────────────────────────────────────
# Runs only when validation FAILS
# Automatically corrects the problematic input
# In a real interactive system, this would ask the USER for corrected input

def fix_input_node(state: State) -> dict:
    a, b, op = state["a"], state["b"], state["operation"]
    print(f"  [fix_input] Error was: '{state['error_msg']}'")
    
    # Fix 1: Unknown operation → default to "add"
    if op not in ("add", "multiply", "divide"):
        op = "add"
        print(f"  [fix_input] Fixed operation: set to 'add'")
    
    # Fix 2: Divide by zero → set b = 1
    if op == "divide" and b == 0:
        b = 1
        print(f"  [fix_input] Fixed b: set to 1 (avoid division by zero)")
    
    # Fix 3: Clamp to valid range
    a = max(-1000, min(1000, a))
    b = max(-1000, min(1000, b))
    
    return {"a": a, "b": b, "operation": op}

print("✓ Fix-input node defined")


In [ ]:
# ── Cell 5: Compute Node ────────────────────────────────────────────────────
def compute_node(state: State) -> dict:
    a, b, op = state["a"], state["b"], state["operation"]
    
    if op == "add":
        res = float(a + b)
        entry = f"{a} + {b} = {res}"
    elif op == "multiply":
        res = float(a * b)
        entry = f"{a} × {b} = {res}"
    else:  # divide (b != 0 guaranteed by validation)
        res = a / b
        entry = f"{a} ÷ {b} = {res:.4f}"
    
    print(f"  [compute]  {entry}")
    return {"result": res, "history": [entry]}

print("✓ Compute node defined")


## The Validation Router — Controls the Loop

```
validate → router ─── "compute"   → compute → END         (valid)
                  └── "fix_input" → fix_input → validate   (invalid, retries left)
                  └── "give_up"   → END                    (max retries reached)
```

The loop forms because:
- `validate` → `fix_input` (when invalid, retries left)  — forward edge
- `fix_input` → `validate` — **BACKWARD EDGE** (creates the cycle!)


In [ ]:
# ── Cell 6: Validation Router ────────────────────────────────────────────────
def validation_router(state: State) -> Literal["compute", "fix_input", "give_up"]:
    if state["is_valid"]:
        return "compute"                          # ✓ Valid → run compute
    elif state["attempts"] >= MAX_ATTEMPTS:
        print(f"  [router] ✗ Max attempts ({MAX_ATTEMPTS}) reached. Giving up.")
        return "give_up"                          # ✗ Too many retries → exit
    else:
        return "fix_input"                        # ✗ Invalid but retry remaining → fix

print("✓ Validation router defined")


In [ ]:
# ── Cell 7: Build Looping Graph ──────────────────────────────────────────────
builder = StateGraph(State)

# Register all nodes
builder.add_node("validate",  validate_node)
builder.add_node("fix_input", fix_input_node)
builder.add_node("compute",   compute_node)

# Entry: always start with validation
builder.add_edge(START, "validate")

# Conditional routing AFTER validate:
builder.add_conditional_edges(
    "validate",          # After validate runs...
    validation_router,   # ...call this router
    {
        "compute":   "compute",    # Valid → run compute
        "fix_input": "fix_input",  # Invalid → go to fix_input
        "give_up":   END,          # Max retries → exit
    }
)

# ── THE CYCLE ──────────────────────────────────────────────────────────────
# fix_input → validate  ← THIS backward edge creates the loop
builder.add_edge("fix_input", "validate")

# compute always exits
builder.add_edge("compute", END)

graph = builder.compile()
print("✓ Looping validation graph compiled")
print("  Cycle: validate → fix_input → validate (until valid or max attempts)")


In [ ]:
# ── Cell 8: Visualize — Notice the Cycle ─────────────────────────────────────
# Look for the backward arrow: fix_input → validate
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 9: Test 1 — Valid Input (No Loop Needed) ────────────────────────────
print("="*55)
print("Test 1: Valid input — should pass on first attempt")
print("="*55)
state = {
    "a": 12, "b": 4, "operation": "divide",
    "result": None, "is_valid": False,
    "error_msg": "", "attempts": 0, "history": []
}
out = graph.invoke(state)
print(f"\n  Result: {out['result']} | Attempts: {out['attempts']}")


In [ ]:
# ── Cell 10: Test 2 — Invalid Operation (Auto-Fixed) ─────────────────────────
print("="*55)
print("Test 2: operation='subtract' → auto-fixed to 'add'")
print("="*55)
state = {
    "a": 8, "b": 3, "operation": "subtract",
    "result": None, "is_valid": False,
    "error_msg": "", "attempts": 0, "history": []
}
out = graph.invoke(state)
print(f"\n  Result: {out['result']} | Attempts: {out['attempts']}")
print(f"  History: {out['history']}")


In [ ]:
# ── Cell 11: Test 3 — Divide by Zero (Auto-Fixed) ────────────────────────────
print("="*55)
print("Test 3: operation='divide', b=0 → b auto-fixed to 1")
print("="*55)
state = {
    "a": 10, "b": 0, "operation": "divide",
    "result": None, "is_valid": False,
    "error_msg": "", "attempts": 0, "history": []
}
out = graph.invoke(state)
print(f"\n  Result: {out['result']} | Attempts: {out['attempts']}")


In [ ]:
# ── Cell 12: Test 4 — Unfixable Error Hits Max Attempts ──────────────────────
# We'll simulate a case where fix_input can't fully fix the problem
# by passing a weird state that keeps failing
# (In practice: bad data that can't be auto-corrected)

print("="*55)
print("Test 4: Customized to show max-attempts exit")
print("="*55)

# Patch fix_input temporarily to NOT fix to show give_up path
def broken_fix(state):
    print(f"  [fix_input] Simulating unfixable: keeping operation='{state['operation']}'")
    return {}  # Does nothing — input stays invalid

builder2 = StateGraph(State)
builder2.add_node("validate",  validate_node)
builder2.add_node("fix_input", broken_fix)    # Broken fix — never actually fixes
builder2.add_node("compute",   compute_node)
builder2.add_edge(START, "validate")
builder2.add_conditional_edges("validate", validation_router,
    {"compute": "compute", "fix_input": "fix_input", "give_up": END})
builder2.add_edge("fix_input", "validate")
builder2.add_edge("compute", END)
graph2 = builder2.compile()

state = {
    "a": 5, "b": 0, "operation": "divide",
    "result": None, "is_valid": False,
    "error_msg": "", "attempts": 0, "history": []
}
out = graph2.invoke(state)
print(f"\n  Result: {out['result']} | Final attempts: {out['attempts']}")
print(f"  Exited via 'give_up' path (max={MAX_ATTEMPTS} reached)")


## Key Concepts Summary

### Creating the Cycle
```python
# Two lines create the loop:
builder.add_edge("fix_input", "validate")      # Backward edge
# + conditional: validate → fix_input (when invalid)
```

### Preventing Infinite Loops — Three Ways

| Method | How |
|--------|-----|
| Exit condition | `is_valid == True` → route to compute |
| Attempt counter | `attempts >= MAX` → route to `give_up` → END |
| LangGraph's safety | `recursion_limit=25` (default) raises `GraphRecursionError` |

### Override recursion limit:
```python
graph.invoke(state, config={"recursion_limit": 50})
```

## Summary

| | Lesson 4 | Lesson 5 |
|--|----------|----------|
| Graph topology | DAG (forward only) | Cyclic (loops allowed) |
| Error handling | Silent / crash | Validate + retry |
| Loop safety | N/A | Attempt counter |
| Input correction | None | fix_input node |

## Limitations → What Lesson 6 Solves
- ❌ `operation` is a hardcoded string — user must pass `"add"`, `"multiply"`, etc.
- ❌ What if the user types `"what is 7 plus 3?"` in natural language?
- ❌ We need an **LLM** to parse natural language → structured operation
- ❌ **Lesson 6**: Vertex AI (Gemini) LLM integration — the first AI-powered lesson
